# Pure-Space Mean-Design Check: Hybrid t-only vs Column t-only

This notebook tests whether richer GLS mean/detrending explains the Hybrid-vs-Column parameter differences in pure space.

Estimated covariance parameters:

- `sigmasq`
- `range_lat`
- `range_lon`
- `nugget`

Mean designs compared:

- `base`: intercept + centered latitude + hour dummies
- `latlon`: intercept + centered latitude + centered longitude + hour dummies
- `hour_spatial`: hour-specific intercept + hour-specific latitude slope + hour-specific longitude slope

Fitting scopes:

- `pooled_day`: the 8 time slots in one day are independent spatial replicates sharing one spatial covariance parameter vector.
- `hourly`: each time slot is fitted separately.

Models compared with m=10:

- `HybridSpace_A10_exactloc`: same-time max-min nearest neighbors, covariance uses `Source_Latitude` / `Source_Longitude`.
- `ColumnSpace_Up2_Right3_M10_realloc`: reverse-L same-time geometry uses regular grid for scan/order, but covariance uses real `Source_Latitude` / `Source_Longitude`.


In [1]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_space_trend_050726 import HybridSpaceTrendVecchiaFit, ColumnSpaceTrendVecchiaFit

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

print('SRC:', SRC)
print('torch:', torch.__version__)
print('device:', DEVICE)


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
torch: 2.5.1
device: cpu


In [2]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]  # 0-based: [2] = July 3
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
MM_COND_NUMBER = 100
FIT_SMOOTHS = [0.5]
MEAN_DESIGNS = ['base', 'latlon', 'hour_spatial']

HYBRID_SPACE_SPEC = {
    'model': 'HybridSpace_A10_exactloc',
    'limit_A': 10,
    'target_chunk_size': 512,
}

COLUMN_SPACE_SPEC = {
    'model_realloc': 'ColumnSpace_Up2_Right3_M10_realloc',
    'head_right_cols': 0,
    'above_count': 2,
    'right_col_count': 3,
    'per_time_conditioning_count': 10,
    'target_chunk_size': 512,
}

LBFGS_LR = 1.0
LBFGS_STEPS = 8
LBFGS_EVAL = 20
LBFGS_HIST = 10

# Variogram-guided, deliberately not tiny nugget.
INIT_SPACE = {
    'sigmasq': 13.0,
    'range_lat': 0.35,
    'range_lon': 0.45,
    'nugget': 2.5,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'nugget']

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'real_pure_space_mean_design_tonly_m10_realloc_050726_results.csv'
OUT_POOLED_CSV = OUT_DIR / 'real_pure_space_mean_design_tonly_m10_realloc_050726_pooled_day.csv'
OUT_HOURLY_CSV = OUT_DIR / 'real_pure_space_mean_design_tonly_m10_realloc_050726_hourly.csv'

print('days:', DAY_IDX_LIST)
print('init:', INIT_SPACE)
print('mean designs:', MEAN_DESIGNS)
print('hybrid:', HYBRID_SPACE_SPEC)
print('column:', COLUMN_SPACE_SPEC)
print('output all:', OUT_CSV)
print('output pooled:', OUT_POOLED_CSV)
print('output hourly:', OUT_HOURLY_CSV)


days: [2]
init: {'sigmasq': 13.0, 'range_lat': 0.35, 'range_lon': 0.45, 'nugget': 2.5}
mean designs: ['base', 'latlon', 'hour_spatial']
hybrid: {'model': 'HybridSpace_A10_exactloc', 'limit_A': 10, 'target_chunk_size': 512}
column: {'model_realloc': 'ColumnSpace_Up2_Right3_M10_realloc', 'head_right_cols': 0, 'above_count': 2, 'right_col_count': 3, 'per_time_conditioning_count': 10, 'target_chunk_size': 512}
output all: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space/log/real_pure_space_mean_design_tonly_m10_realloc_050726_results.csv
output pooled: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space/log/real_pure_space_mean_design_tonly_m10_realloc_050726_pooled_day.csv
output hourly: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space/log/real_pure_space_mean_design_tonly_m10_realloc_050726_hourly.csv


In [3]:
def phys_to_log_space(d):
    return [np.log(d['sigmasq']), np.log(d['range_lat']), np.log(d['range_lon']), np.log(d['nugget'])]


def backmap_space(out_params):
    return {
        'sigmasq': float(np.exp(out_params[0])),
        'range_lat': float(np.exp(out_params[1])),
        'range_lon': float(np.exp(out_params[2])),
        'nugget': float(np.exp(out_params[3])),
    }


def make_space_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log_space(INIT_SPACE)]


def count_valid(day_map):
    total = 0
    for v in day_map.values():
        ok = (~torch.isnan(v[:, 2])) & (~torch.isnan(v[:, 0])) & (~torch.isnan(v[:, 1]))
        total += int(ok.sum().item())
    return total


def empty_device_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def save_results(rows):
    df = pd.DataFrame(rows)
    round_df(df).to_csv(OUT_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    if 'fit_scope' in df.columns:
        pooled = df[df['fit_scope'] == 'pooled_day'].copy()
        hourly = df[df['fit_scope'] == 'hourly'].copy()
        if len(pooled):
            round_df(pooled).to_csv(OUT_POOLED_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
        if len(hourly):
            round_df(hourly).to_csv(OUT_HOURLY_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    return df


def space_diag(model):
    groups = getattr(model, 'Batched_Groups', [])
    if not groups:
        return {
            'n_batches': 0,
            'n_tails': 0,
            'mean_m': 0.0,
            'max_m': 0,
            'largest_batch_n': 0,
        }
    batch_ns = np.asarray([int(g['target_idx'].shape[0]) for g in groups], dtype=np.int64)
    m_sizes = np.asarray([int(g['max_m']) for g in groups], dtype=np.int64)
    return {
        'n_batches': int(len(groups)),
        'n_tails': int(sum(batch_ns)),
        'mean_m': float(m_sizes.mean()),
        'max_m': int(m_sizes.max()),
        'largest_batch_n': int(batch_ns.max()),
    }


def result_row(date_str, day_idx, smooth, mean_design, model_name, kernel, coord_mode, loss, fit_iter, pre_s, fit_s, n_valid, est, diag, fit_scope, hour_idx=None, time_key=None):
    row = {
        'date_str': date_str,
        'day_idx': int(day_idx),
        'hour_idx': np.nan if hour_idx is None else int(hour_idx),
        'time_key': '' if time_key is None else str(time_key),
        'fit_scope': fit_scope,
        'smooth': float(smooth),
        'mean_design': mean_design,
        'model': model_name,
        'kernel': kernel,
        'coord_mode': coord_mode,
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': round(pre_s, 3),
        'fit_s': round(fit_s, 3),
        'total_s': round(pre_s + fit_s, 3),
        'n_valid': int(n_valid),
    }
    row.update({f'est_{k}': est[k] for k in P_LABELS})
    row.update(diag)
    return row

print('Initial log params:', [round(x, 4) for x in phys_to_log_space(INIT_SPACE)])


Initial log params: [2.5649, -1.0498, -0.7985, 0.9163]


In [4]:
# Load full month once, then trim to one day.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_month_keys = sorted(df_map.keys())
day_key_map = {
    day_idx: sorted_month_keys[day_idx * 8:(day_idx + 1) * 8]
    for day_idx in DAY_IDX_LIST
}
selected_keys = [k for day_idx in DAY_IDX_LIST for k in day_key_map[day_idx]]
df_map_selected = {k: df_map[k] for k in selected_keys}

first_key = selected_keys[0]
first_df = df_map_selected[first_key].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)
source_coords_t0_ordered = first_df[['Source_Latitude', 'Source_Longitude']].values.astype(np.float64)

del df_map
gc.collect()

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Month time slots loaded then trimmed: {len(sorted_month_keys)} -> {len(selected_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')
print('grid lat:', grid_coords_ordered[:, 0].min(), grid_coords_ordered[:, 0].max())
print('grid lon:', grid_coords_ordered[:, 1].min(), grid_coords_ordered[:, 1].max())
print('source selected t0 lat:', np.nanmin(source_coords_t0_ordered[:, 0]), np.nanmax(source_coords_t0_ordered[:, 0]))
print('source selected t0 lon:', np.nanmin(source_coords_t0_ordered[:, 1]), np.nanmax(source_coords_t0_ordered[:, 1]))


--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
Monthly mean TCO: 257.9726
Month time slots loaded then trimmed: 248 -> 8
Grid cells: 18,126
grid lat: -2.9720000000000044 2.0
grid lon: 121.04600000000188 131.0
source selected t0 lat: -2.9828918 1.9999858
source selected t0 lon: 121.0146 130.99957


In [5]:
def load_map_for_keys(keys, keep_ori):
    day_df_map = {k: df_map_selected[k] for k in keys}
    hour_indices = [0, len(keys)]
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=keep_ori,
    )
    return {k: v.to(DEVICE) for k, v in day_map.items()}


def keys_for_scope(day_idx, fit_scope, hour_idx=None):
    day_keys = day_key_map[day_idx]
    if fit_scope == 'pooled_day':
        return day_keys, f'{YEAR}{MONTH:02d}{day_idx + 1:02d}', None, None
    if fit_scope == 'hourly':
        key = day_keys[int(hour_idx)]
        return [key], f'{YEAR}{MONTH:02d}{day_idx + 1:02d}_h{int(hour_idx) + 1:02d}', int(hour_idx), key
    raise ValueError(f'Unknown fit_scope={fit_scope}')


def fit_hybrid_space(day_idx, smooth, mean_design, fit_scope, hour_idx=None):
    keys, date_str, hour_idx_out, time_key = keys_for_scope(day_idx, fit_scope, hour_idx)
    day_map = load_map_for_keys(keys, keep_ori=True)
    n_valid = count_valid(day_map)
    print('\n' + '=' * 80)
    print(f"HYBRID SPACE | mean={mean_design} | scope={fit_scope} | {date_str} | smooth={smooth} | valid={n_valid:,}")

    params = make_space_params()
    model = HybridSpaceTrendVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        nns_map=nns_map_full,
        limit_A=HYBRID_SPACE_SPEC['limit_A'],
        target_chunk_size=HYBRID_SPACE_SPEC['target_chunk_size'],
        mean_design=mean_design,
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = space_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    est = backmap_space(out)

    row = result_row(
        date_str, day_idx, smooth, mean_design, HYBRID_SPACE_SPEC['model'], 'hybrid_space_tonly',
        'Source_Latitude/Source_Longitude', float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est, diag,
        fit_scope=fit_scope, hour_idx=hour_idx_out, time_key=time_key,
    )
    row.update({'total_conditioning_nominal': HYBRID_SPACE_SPEC['limit_A']})
    print('RESULT:', row)
    del model, day_map, params, opt
    empty_device_cache()
    return row


def fit_column_space(day_idx, smooth, mean_design, coord_mode, fit_scope, hour_idx=None):
    keys, date_str, hour_idx_out, time_key = keys_for_scope(day_idx, fit_scope, hour_idx)
    keep_ori = True
    model_name = COLUMN_SPACE_SPEC['model_realloc']
    day_map = load_map_for_keys(keys, keep_ori=keep_ori)
    n_valid = count_valid(day_map)
    print('\n' + '=' * 80)
    print(f"COLUMN SPACE | mean={mean_design} | scope={fit_scope} | {model_name} | {date_str} | smooth={smooth} | valid={n_valid:,}")

    params = make_space_params()
    model = ColumnSpaceTrendVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        grid_coords=grid_coords_ordered,
        head_right_cols=COLUMN_SPACE_SPEC['head_right_cols'],
        above_count=COLUMN_SPACE_SPEC['above_count'],
        right_col_count=COLUMN_SPACE_SPEC['right_col_count'],
        per_time_conditioning_count=COLUMN_SPACE_SPEC['per_time_conditioning_count'],
        target_chunk_size=COLUMN_SPACE_SPEC['target_chunk_size'],
        mean_design=mean_design,
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = space_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    est = backmap_space(out)

    row = result_row(
        date_str, day_idx, smooth, mean_design, model_name, 'column_space_tonly',
        'Source_Latitude/Source_Longitude',
        float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est, diag,
        fit_scope=fit_scope, hour_idx=hour_idx_out, time_key=time_key,
    )
    row.update({
        'total_conditioning_nominal': COLUMN_SPACE_SPEC['per_time_conditioning_count'],
        'head_right_cols': COLUMN_SPACE_SPEC['head_right_cols'],
        'above_count': COLUMN_SPACE_SPEC['above_count'],
        'right_col_count': COLUMN_SPACE_SPEC['right_col_count'],
        'per_time_conditioning_count': COLUMN_SPACE_SPEC['per_time_conditioning_count'],
    })
    print('RESULT:', row)
    del model, day_map, params, opt
    empty_device_cache()
    return row


In [6]:
# Run pure-space mean-design comparison: pooled day and each hour separately.
rows = []
for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        for mean_design in MEAN_DESIGNS:
            # One parameter vector shared by all 8 same-day time slots.
            rows.append(fit_hybrid_space(day_idx, smooth, mean_design, fit_scope='pooled_day'))
            save_results(rows)
            rows.append(fit_column_space(day_idx, smooth, mean_design, coord_mode='realloc', fit_scope='pooled_day'))
            save_results(rows)

            # One parameter vector per time slot.
            for hour_idx in range(len(day_key_map[day_idx])):
                rows.append(fit_hybrid_space(day_idx, smooth, mean_design, fit_scope='hourly', hour_idx=hour_idx))
                save_results(rows)
                rows.append(fit_column_space(day_idx, smooth, mean_design, coord_mode='realloc', fit_scope='hourly', hour_idx=hour_idx))
                save_results(rows)

results = pd.DataFrame(rows)
print('Saved all:', OUT_CSV)
print('Saved pooled:', OUT_POOLED_CSV)
print('Saved hourly:', OUT_HOURLY_CSV)
display(round_df(results))



HYBRID SPACE | mean=base | scope=pooled_day | 20240703 | smooth=0.5 | valid=140,352
Pre-computing HybridSpaceVecchia [A=10]... Done in 0.5s. tails=140352, m mean/med/max=10.0/10/10
--- Starting Pure-Space Vecchia L-BFGS ---
--- Step 1/8 / Loss: 1.402569 / Max Grad: 6.68e-06 ---
Converged: max_grad 6.68e-06 < 1.00e-05
Final Pure-Space Params: {'sigmasq': 14.822116050442943, 'range_lat': 0.20309257231715697, 'range_lon': 0.2366832343929837, 'nugget': 1.641035777175173}
RESULT: {'date_str': '20240703', 'day_idx': 2, 'hour_idx': nan, 'time_key': '', 'fit_scope': 'pooled_day', 'smooth': 0.5, 'mean_design': 'base', 'model': 'HybridSpace_A10_exactloc', 'kernel': 'hybrid_space_tonly', 'coord_mode': 'Source_Latitude/Source_Longitude', 'loss': 1.4025693915326158, 'fit_iter_raw': 0, 'fit_steps_reported': 1, 'precompute_s': 0.507, 'fit_s': 15.467, 'total_s': 15.974, 'n_valid': 140352, 'est_sigmasq': 14.822116050442943, 'est_range_lat': 0.20309257231715697, 'est_range_lon': 0.2366832343929837, 'es

,date_str,day_idx,hour_idx,time_key,fit_scope,smooth,mean_design,model,kernel,coord_mode,...,n_batches,n_tails,mean_m,max_m,largest_batch_n,total_conditioning_nominal,head_right_cols,above_count,right_col_count,per_time_conditioning_count
0,20240703,2,NaN,,pooled_day,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,...,1,140352,10.0,10,140352,10,NaN,NaN,NaN,NaN
1,20240703,2,NaN,,pooled_day,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,...,1,140352,10.0,10,140352,10,0.0,2.0,3.0,10.0
2,20240703_h01,2,0.0,2024_07_y24m07day03_hm00:53,hourly,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,...,1,18067,10.0,10,18067,10,NaN,NaN,NaN,NaN
3,20240703_h01,2,0.0,2024_07_y24m07day03_hm00:53,hourly,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,...,1,18067,10.0,10,18067,10,0.0,2.0,3.0,10.0
4,20240703_h02,2,1.0,2024_07_y24m07day03_hm01:53,hourly,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,...,1,17670,10.0,10,17670,10,NaN,NaN,NaN,NaN
5,20240703_h02,2,1.0,2024_07_y24m07day03_hm01:53,hourly,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,...,1,17670,10.0,10,17670,10,0.0,2.0,3.0,10.0
6,20240703_h03,2,2.0,2024_07_y24m07day03_hm02:53,hourly,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,...,1,16698,10.0,10,16698,10,NaN,NaN,NaN,NaN
7,20240703_h03,2,2.0,2024_07_y24m07day03_hm02:53,hourly,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,...,1,16698,10.0,10,16698,10,0.0,2.0,3.0,10.0
8,20240703_h04,2,3.0,2024_07_y24m07day03_hm03:53,hourly,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,...,1,16863,10.0,10,16863,10,NaN,NaN,NaN,NaN
9,20240703_h04,2,3.0,2024_07_y24m07day03_hm03:53,hourly,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,...,1,16863,10.0,10,16863,10,0.0,2.0,3.0,10.0


In [7]:
summary_cols = [
    'date_str', 'fit_scope', 'hour_idx', 'time_key', 'smooth', 'mean_design', 'model', 'kernel', 'coord_mode', 'total_conditioning_nominal',
    'loss', 'precompute_s', 'fit_s', 'total_s', 'n_valid', 'n_batches', 'n_tails',
    'mean_m', 'max_m', 'largest_batch_n',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_nugget',
]
existing = [c for c in summary_cols if c in results.columns]
display(round_df(results[existing].sort_values(['fit_scope', 'mean_design', 'date_str', 'hour_idx', 'model'])))

param_rows = []
for _, row in results.iterrows():
    for p in P_LABELS:
        param_rows.append({
            'date_str': row['date_str'],
            'fit_scope': row.get('fit_scope', ''),
            'hour_idx': row.get('hour_idx', np.nan),
            'time_key': row.get('time_key', ''),
            'mean_design': row.get('mean_design', ''),
            'model': row['model'],
            'parameter': p,
            'estimate': row[f'est_{p}'],
        })
param_est = pd.DataFrame(param_rows)
display(round_df(param_est))


,date_str,fit_scope,hour_idx,time_key,smooth,mean_design,model,kernel,coord_mode,total_conditioning_nominal,...,n_valid,n_batches,n_tails,mean_m,max_m,largest_batch_n,est_sigmasq,est_range_lat,est_range_lon,est_nugget
3,20240703_h01,hourly,0.0,2024_07_y24m07day03_hm00:53,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,10,...,18067,1,18067,10.0,10,18067,11.8275,0.2371,0.3362,2.9654
2,20240703_h01,hourly,0.0,2024_07_y24m07day03_hm00:53,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,10,...,18067,1,18067,10.0,10,18067,11.5267,0.2753,0.3831,3.2931
5,20240703_h02,hourly,1.0,2024_07_y24m07day03_hm01:53,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,10,...,17670,1,17670,10.0,10,17670,13.0262,0.1976,0.2882,1.8309
4,20240703_h02,hourly,1.0,2024_07_y24m07day03_hm01:53,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,10,...,17670,1,17670,10.0,10,17670,13.8899,0.2274,0.3157,1.9319
7,20240703_h03,hourly,2.0,2024_07_y24m07day03_hm02:53,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,10,...,16698,1,16698,10.0,10,16698,15.5766,0.1970,0.2608,1.2124
6,20240703_h03,hourly,2.0,2024_07_y24m07day03_hm02:53,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,10,...,16698,1,16698,10.0,10,16698,16.2563,0.2161,0.2799,1.2985
9,20240703_h04,hourly,3.0,2024_07_y24m07day03_hm03:53,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,10,...,16863,1,16863,10.0,10,16863,17.2768,0.1847,0.2354,0.8615
8,20240703_h04,hourly,3.0,2024_07_y24m07day03_hm03:53,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,10,...,16863,1,16863,10.0,10,16863,17.6478,0.2084,0.2627,1.1280
11,20240703_h05,hourly,4.0,2024_07_y24m07day03_hm04:48,0.5,base,ColumnSpace_Up2_Right3_M10_realloc,column_space_tonly,Source_Latitude/Source_Longitude,10,...,17088,1,17088,10.0,10,17088,17.3970,0.1730,0.2116,0.9281
10,20240703_h05,hourly,4.0,2024_07_y24m07day03_hm04:48,0.5,base,HybridSpace_A10_exactloc,hybrid_space_tonly,Source_Latitude/Source_Longitude,10,...,17088,1,17088,10.0,10,17088,17.9760,0.1884,0.2267,1.0433


,date_str,fit_scope,hour_idx,time_key,mean_design,model,parameter,estimate
0,20240703,pooled_day,NaN,,base,HybridSpace_A10_exactloc,sigmasq,14.8221
1,20240703,pooled_day,NaN,,base,HybridSpace_A10_exactloc,range_lat,0.2031
2,20240703,pooled_day,NaN,,base,HybridSpace_A10_exactloc,range_lon,0.2367
3,20240703,pooled_day,NaN,,base,HybridSpace_A10_exactloc,nugget,1.6410
4,20240703,pooled_day,NaN,,base,ColumnSpace_Up2_Right3_M10_realloc,sigmasq,14.3775
...,...,...,...,...,...,...,...,...
211,20240703_h08,hourly,7.0,2024_07_y24m07day03_hm07:48,hour_spatial,HybridSpace_A10_exactloc,nugget,1.8556
212,20240703_h08,hourly,7.0,2024_07_y24m07day03_hm07:48,hour_spatial,ColumnSpace_Up2_Right3_M10_realloc,sigmasq,11.9404
213,20240703_h08,hourly,7.0,2024_07_y24m07day03_hm07:48,hour_spatial,ColumnSpace_Up2_Right3_M10_realloc,range_lat,0.1439
214,20240703_h08,hourly,7.0,2024_07_y24m07day03_hm07:48,hour_spatial,ColumnSpace_Up2_Right3_M10_realloc,range_lon,0.1517
